<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_MLR/17_2_4_1_MLR_Ames_Part1_Revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLR Predicting Housing Prices in Ames Iowa: Part 1,  Data Cleaning

Author: Brad Sheese
___

## Purpose of this Notebook
This notebook demonstrates manual cleaning of a real-world dataset. The intent of the cleaning is to prepare the dataset for inference, wherein we are interested in examining individual coefficients of features in the model.

### Data Cleaning, Under and Over Cleaning
The primary goal of data cleaning is to transform messy, raw data into a structured, model-ready format that maximizes true predictive signals and minimizes disruptive noise. This involves transforming the data in different ways. The trick is that we need to do this  without distorting the underlying reality of the information.

Achieving this balance requires navigating the dual pitfalls of under-cleaning and over-cleaning. Under-cleaning leaves too much chaos in the dataset, such as unhandled missing values, unencoded strings, extreme outliers, or highly collinear features, which can cause machine learning algorithms to crash, fail to converge, or overfit on structural errors. Conversely, over-cleaning stems from an overzealous attempt to simplify or shrink the data, resulting in the destruction of valuable information through the aggressive dropping of features, the clumsy grouping of highly distinct categories into generic buckets, or inappropriate binarization of continuous variables.

### Cleaning Specifically with MLR in Mind
Preparing data specifically for multiple linear regression (MLR)  requires strict adherence to the algorithm’s statistical assumptions.
* Because MLR relies on ordinary least squares (OLS) optimization, it is highly sensitive to missing values and extreme outliers, both of which can disproportionately pull the line of best fit and must be carefully imputed, capped, or removed.
* Since the MLR equation requires strictly numeric inputs, categorical data must be converted into binary flags using one-hot encoding, with the critical step of dropping one baseline category to avoid the "dummy variable trap" (perfect multicollinearity).
* Mitigating multicollinearity is a primary focus of MLR
cleaning. We routinely use Variance Inflation Factor (VIF) scores to
identify and remove highly correlated overlapping predictors that would
otherwise cause the model's coefficient estimates to become wildly unstable and
meaningless.
* Applying mathematical transformations (such as logarithmic
scaling) to highly skewed variables helps satisfy the strict assumptions of
linearity and homoscedasticity, ensuring the final model is not just
mathematically functional, but statistically valid and interpretable.

---


## Data for this Exercise
**Data Source:** http://jse.amstat.org/v19n3/decock/AmesHousing.txt


The Ames housing dataset, compiled by Dean De Cock in 2011, is a comprehensive collection of information detailing nearly every aspect of residential property sales in Ames, Iowa, from 2006 to 2010. It consists of approximately 2,930 observations and 79 explanatory variables, ranging from house quality and neighborhood to square footage and number of bathrooms, which are used to predict the final "SalePrice" of each home.

## Get the Data, Examine the Dataframe

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Data source
url = 'https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_housing_ames.txt'

# Loading the dataframe
df_raw = pd.read_csv(url, sep='\t')

# Initial cleaning: remove extreme outliers per the author's recommendation, large houses sold for little money due to inheritance
df = df_raw.loc[df_raw['Gr Liv Area'] < 4000, :].copy()

# helper functions
def safe_drop(df: pd.DataFrame, drop_list: list[str]) -> pd.DataFrame:
  """
  Safely drops columns from a Pandas DataFrame if they exist.

  Args:
    df (pd.DataFrame): The input DataFrame.
    drop_list (list[str]): A list of column names to attempt to drop.

  Returns:
    pd.DataFrame: The DataFrame with specified columns dropped (if they existed).
  """
  # Filter drop_list to only include columns that exist in the DataFrame
  existing_cols_to_drop = [col for col in drop_list if col in df.columns]

  # Drop the columns if there are any to drop
  if existing_cols_to_drop:
    df = df.drop(existing_cols_to_drop, axis=1)
  return df

df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


We use `.info()` to examine our dataframe. Look closely at the "Non-Null Count."


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2925 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2925 non-null   int64  
 1   PID              2925 non-null   int64  
 2   MS SubClass      2925 non-null   int64  
 3   MS Zoning        2925 non-null   object 
 4   Lot Frontage     2435 non-null   float64
 5   Lot Area         2925 non-null   int64  
 6   Street           2925 non-null   object 
 7   Alley            198 non-null    object 
 8   Lot Shape        2925 non-null   object 
 9   Land Contour     2925 non-null   object 
 10  Utilities        2925 non-null   object 
 11  Lot Config       2925 non-null   object 
 12  Land Slope       2925 non-null   object 
 13  Neighborhood     2925 non-null   object 
 14  Condition 1      2925 non-null   object 
 15  Condition 2      2925 non-null   object 
 16  Bldg Type        2925 non-null   object 
 17  House Style      29

We have a lot of features with missing values and a lot of features that are classified as 'objects'. There's quite a bit of cleanup we need to do.

## Remove Uninformative Features
Order and PID are simply unique identifiers and will not provide anything usefule for our model. We can remove these from our model.

Note: `safe_drop()` is used to check for the existence of a column before we attempt to drop it. This is helpful to do when working in notebooks where we might re-run a single cell multiple times.  

In [4]:
print(df.shape)
df = safe_drop(df, ['Order', 'PID'])
print(df.shape)

(2925, 82)
(2925, 80)


Let's look for columns that are monotonic (only contain a single value). These would also be uninformative and could be dropped.  

In [5]:
monotonic_list = [x for x in df.columns if len(df[x].unique()) <= 1]
monotonic_list

[]

Look for duplicate rows and rows where all values are missing.

In [6]:
print(f"Number of duplicate rows: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Number of rows after dropping duplicates: {df.shape[0]}\n")

print(f"Number of rows with all NaNs: {df.isna().all(axis=1).sum()}")
df = df.dropna(how='all')
print(f"Number of rows after dropping all-NaN rows: {df.shape[0]}")

Number of duplicate rows: 0
Number of rows after dropping duplicates: 2925

Number of rows with all NaNs: 0
Number of rows after dropping all-NaN rows: 2925


## Yolked Variables

We are looking for "yolked features" where the value of one feature is dependent upon the value of another. The most common type of yolked feature here is where a house feature is missing (coded as None, hopefully) and the numeric values describing that missing feature is zero.

The code to automate the search for these features is not included in this notebook. You do not need to look at it or know how it works. The analysis identified 38 "yolked feature" pairs, where the 'None' category in a one-hot encoded categorical feature consistently corresponds to a 0 value in a related numerical feature.
*   **Masonry Veneer:** When 'Mas Vnr Type' is 'None', 'Mas Vnr Area' is consistently 0.
*   **Basement Features:** All basement-related categorical features (`Bsmt Qual`, `Bsmt Cond`, `Bsmt Exposure`, `BsmtFin Type 1`, `BsmtFin Type 2`) are yolked with all basement area numerical features (`BsmtFinSF1`, `BsmtFinSF2`, `Bsmt Unf SF`, `Total Bsmt SF`). This means if any of these categorical basement features is 'None' (indicating no basement), then all basement area measurements are 0.
*   **Garage Features:** All garage-related categorical features (`Garage Type`, `Garage Finish`, `Garage Qual`, `Garage Cond`) are yolked with numerical features describing the garage (`Garage Yr Blt`, `Garage Cars`, `Garage Area`). This implies that if a house has no garage (categorical 'None'), then its garage-related numerical attributes are 0.
*   **Fireplace Features:** 'Fireplace Qu' is yolked with 'Fireplaces'.
*   **Misc Features:** 'Misc Feature' is yolked with 'Misc Val'.
*   **Pool Features:** 'Pool QC' is yolked with 'Pool Area'.

*   **Other Yolked Pairs:** These have probably been erroneously idenfied by our code.
    *   'Electrical' is yolked with 'Garage Area'.
    *   'Fence' is yolked with 'Wood Deck SF'.
    

### Resolving the Yolked Variable Issue
One way to think about this problem is that we have information shared across two features, and what we'd like to do is to have each feature be relatively independent of one another.

In [8]:
print(df.shape)

### Pool and Misc Features:
# Drop these entirely (both categorical and numerical).
# 99% of houses don't have pools or "misc" features.
# Keeping them adds noise with almost no predictive value for a beginner model.

df = safe_drop(df, ['Pool QC', 'Pool Area', 'Misc Feature', 'Misc Val'])

### Garage Yr Blt:
# If a house has no garage, the year built is often coded as 0 or NA.
# A year of 0 will act as a massive outlier.
# I'd be tempted to do a median replacement, let's drop to keep things simpler for a beginner model.
df = safe_drop(df, ['Garage Yr Blt'])

### Other drops:
# To try to make this exercise shorter, I'm not going to explain these.
more_drops_list = ['Alley', 'Fence', 'Mas Vnr Type', 'Bsmt Qual', 'Bsmt Cond', \
                   'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Fireplace Qu', \
                   'Neighborhood', 'MS Subclass', 'Mo Sold', 'Kitchen Qual', 'Exter Qual', \
                   'Heating QC']
df = safe_drop(df, more_drops_list)

print(df.shape)


(2925, 80)
(2925, 61)


In [9]:
# lets look at garage related features in more detail
garage_list = [col_name for col_name in df.columns if 'garage' in col_name.lower()]

print(df[garage_list].info(), '\n')

# explore garage categoricals
for col in df[garage_list].select_dtypes(include=['object']).columns:
  print(f"{col}: {df[col].value_counts(normalize=True, dropna=False)}\n")

# garage qual and garage cond are 90% a single value, let's drop these
df = safe_drop(df, ['Garage Qual', 'Garage Cond'])

# garage type has some very uncommon values lets collapse into a binary
if 'garage_attached' not in df.columns:
  df['garage_attached'] = np.where(df['Garage Type'] == 'Attchd', 1, 0)

# same for garage unfinished, let's collapse
if 'garage_unfinished' not in df.columns:
  df['garage_unfinished'] = np.where(df['Garage Finish'] == 'Unf', 1, 0)

df = safe_drop(df, ['Garage Type', 'Garage Finish'])



<class 'pandas.core.frame.DataFrame'>
Index: 2925 entries, 0 to 2929
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Garage Type    2768 non-null   object 
 1   Garage Finish  2766 non-null   object 
 2   Garage Cars    2924 non-null   float64
 3   Garage Area    2924 non-null   float64
 4   Garage Qual    2766 non-null   object 
 5   Garage Cond    2766 non-null   object 
dtypes: float64(2), object(4)
memory usage: 160.0+ KB
None 

Garage Type: Garage Type
Attchd     0.590427
Detchd     0.267350
BuiltIn    0.063248
NaN        0.053675
Basment    0.012308
2Types     0.007863
CarPort    0.005128
Name: proportion, dtype: float64

Garage Finish: Garage Finish
Unf    0.420855
RFn    0.277607
Fin    0.247179
NaN    0.054359
Name: proportion, dtype: float64

Garage Qual: Garage Qual
TA     0.892308
NaN    0.054359
Fa     0.042393
Gd     0.008205
Po     0.001709
Ex     0.001026
Name: proportion, dtype: float64

Garag

## Cleaning Categoricals

In [10]:
# let's look for unbalanced categoricals, starting with features where
# one category accounts for more than 70% or the values
for col in df.select_dtypes(include=['object']).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.70:
    print(df[col].value_counts(normalize=True))
    print()

unbalanced_categorical_drop_list = ['Street', 'Land Contour', 'Utilities', \
                                    'Land Slope', 'Condition 1', 'Condition 2',\
                                    'Roof Matl', 'Exter Cond', 'Heating', 'Central Air', \
                                    'Electrical', 'Functional', 'Paved Drive', "Sale Type"]

df = safe_drop(df, unbalanced_categorical_drop_list)

MS Zoning
RL         0.775385
RM         0.157949
FV         0.047521
RH         0.009231
C (all)    0.008547
I (all)    0.000684
A (agr)    0.000684
Name: proportion, dtype: float64

Street
Pave    0.995897
Grvl    0.004103
Name: proportion, dtype: float64

Land Contour
Lvl    0.899487
HLS    0.041026
Bnk    0.038974
Low    0.020513
Name: proportion, dtype: float64

Utilities
AllPub    0.998974
NoSewr    0.000684
NoSeWa    0.000342
Name: proportion, dtype: float64

Lot Config
Inside     0.730940
Corner     0.173675
CulDSac    0.061538
FR2        0.029060
FR3        0.004786
Name: proportion, dtype: float64

Land Slope
Gtl    0.951795
Mod    0.042735
Sev    0.005470
Name: proportion, dtype: float64

Condition 1
Norm      0.861197
Feedr     0.055726
Artery    0.031453
RRAn      0.017094
PosN      0.012991
RRAe      0.009573
PosA      0.006838
RRNn      0.003077
RRNe      0.002051
Name: proportion, dtype: float64

Condition 2
Norm      0.990085
Feedr     0.004444
Artery    0.001709
PosA 

In [11]:
# let's broaden the search a bit to look at above 50%
for col in df.select_dtypes(include=['object']).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.50:
    print(df[col].value_counts(normalize=True))
    print()
    # create a new feature that binary encodes the top feature versus all others
    top_value = (df[col].value_counts(normalize=True, dropna=False).index[0])
    df[col + '_' + top_value] = np.where(df[col] == top_value, 1, 0)
    df = safe_drop(df, [col])


MS Zoning
RL         0.775385
RM         0.157949
FV         0.047521
RH         0.009231
C (all)    0.008547
I (all)    0.000684
A (agr)    0.000684
Name: proportion, dtype: float64

Lot Shape
Reg    0.635556
IR1    0.333333
IR2    0.025983
IR3    0.005128
Name: proportion, dtype: float64

Lot Config
Inside     0.730940
Corner     0.173675
CulDSac    0.061538
FR2        0.029060
FR3        0.004786
Name: proportion, dtype: float64

Bldg Type
1Fam      0.827350
TwnhsE    0.079658
Duplex    0.037265
Twnhs     0.034530
2fmCon    0.021197
Name: proportion, dtype: float64

House Style
1Story    0.505983
2Story    0.297094
1.5Fin    0.107350
SLvl      0.043761
SFoyer    0.028376
2.5Unf    0.008205
1.5Unf    0.006496
2.5Fin    0.002735
Name: proportion, dtype: float64

Roof Style
Gable      0.793162
Hip        0.187009
Gambrel    0.007521
Flat       0.006838
Mansard    0.003761
Shed       0.001709
Name: proportion, dtype: float64

Sale Condition
Normal     0.824615
Partial    0.082735
Abnorm

In [12]:
# Let's see what remains
for col in df.select_dtypes(include=['object']).columns:
  print(df[col].value_counts(normalize=True))
  print()

# Convert Foundation to Three Categories
df.loc[~df['Foundation'].isin(['PConc', 'CBlock']), 'Foundation'] = 'Other'

# I have no good plan for the exteriors, and I don't want to explode the one_hots
# so lets drop
df = safe_drop(df, ['Exterior 1st', 'Exterior 2nd'])

Exterior 1st
VinylSd    0.350769
MetalSd    0.153846
HdBoard    0.150769
Wd Sdng    0.143248
Plywood    0.075556
CemntBd    0.042393
BrkFace    0.030085
WdShing    0.019145
AsbShng    0.015043
Stucco     0.014359
BrkComm    0.002051
AsphShn    0.000684
CBlock     0.000684
Stone      0.000684
PreCast    0.000342
ImStucc    0.000342
Name: proportion, dtype: float64

Exterior 2nd
VinylSd    0.347009
MetalSd    0.152821
HdBoard    0.138462
Wd Sdng    0.135726
Plywood    0.093675
CmentBd    0.042393
Wd Shng    0.027692
BrkFace    0.016068
Stucco     0.015726
AsbShng    0.012991
Brk Cmn    0.007521
ImStucc    0.004786
Stone      0.002051
AsphShn    0.001368
CBlock     0.001026
PreCast    0.000342
Other      0.000342
Name: proportion, dtype: float64

Foundation
PConc     0.446154
CBlock    0.425299
BrkTil    0.106325
Slab      0.016752
Stone     0.003761
Wood      0.001709
Name: proportion, dtype: float64



In [13]:
# verify foundation and convert to category
for col in df.select_dtypes('object').columns:
  print(col, '\n', df[col].value_counts(dropna=False, normalize=True), '\n')
  df[col] = df[col].astype('category')

Foundation 
 Foundation
PConc     0.446154
CBlock    0.425299
Other     0.128547
Name: proportion, dtype: float64 



## Cleaning Numerics

In [14]:
#recheck the current status
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2925 entries, 0 to 2929
Data columns (total 43 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   MS SubClass            2925 non-null   int64   
 1   Lot Frontage           2435 non-null   float64 
 2   Lot Area               2925 non-null   int64   
 3   Overall Qual           2925 non-null   int64   
 4   Overall Cond           2925 non-null   int64   
 5   Year Built             2925 non-null   int64   
 6   Year Remod/Add         2925 non-null   int64   
 7   Mas Vnr Area           2902 non-null   float64 
 8   Foundation             2925 non-null   category
 9   BsmtFin SF 1           2924 non-null   float64 
 10  BsmtFin SF 2           2924 non-null   float64 
 11  Bsmt Unf SF            2924 non-null   float64 
 12  Total Bsmt SF          2924 non-null   float64 
 13  1st Flr SF             2925 non-null   int64   
 14  2nd Flr SF             2925 non-null   int64 

### Identify Binaries and Switch to Bool
Some of our features of the 'object' datatype can be converted to the boolean datatype. This is will speed up some of our analyses.

In [15]:
# Use the newer nullable "boolean" type which supports pd.NA
for col in df.select_dtypes('object').columns:
    # Check if exactly 2 values EXIST (excluding NA)
    if df[col].nunique(dropna=True) == 2:
        # Map values to True/False explicitly to avoid 'any non-zero is True' logic
        # and use 'boolean' (capital B) to preserve NAs
        unique_vals = df[col].dropna().unique()
        df[col] = df[col].map({unique_vals[0]: True, unique_vals[1]: False}).astype('boolean')

# Now this will correctly show your newly converted nullable booleans
df.select_dtypes('boolean').info()

<class 'pandas.core.frame.DataFrame'>
Index: 2925 entries, 0 to 2929
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   garage_attached        2925 non-null   bool 
 1   garage_unfinished      2925 non-null   bool 
 2   MS Zoning_RL           2925 non-null   bool 
 3   Lot Shape_Reg          2925 non-null   bool 
 4   Lot Config_Inside      2925 non-null   bool 
 5   Bldg Type_1Fam         2925 non-null   bool 
 6   House Style_1Story     2925 non-null   bool 
 7   Roof Style_Gable       2925 non-null   bool 
 8   Sale Condition_Normal  2925 non-null   bool 
dtypes: bool(9)
memory usage: 48.6 KB


### Unbalanced Numerics

In [16]:
# look for 'unbalanced' numerics, greater than 90%, drop them
for col in df.select_dtypes(include=np.number).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.90:
    print(df[col].value_counts(normalize=True))
    print()
    df = safe_drop(df, [col])

Low Qual Fin SF
0       0.986325
80      0.001368
360     0.000684
205     0.000684
144     0.000342
362     0.000342
390     0.000342
431     0.000342
120     0.000342
436     0.000342
371     0.000342
1064    0.000342
259     0.000342
397     0.000342
513     0.000342
312     0.000342
108     0.000342
156     0.000342
697     0.000342
232     0.000342
420     0.000342
384     0.000342
512     0.000342
473     0.000342
114     0.000342
479     0.000342
515     0.000342
528     0.000342
53      0.000342
392     0.000342
572     0.000342
234     0.000342
140     0.000342
450     0.000342
481     0.000342
514     0.000342
Name: proportion, dtype: float64

Bsmt Half Bath
0.0    0.940814
1.0    0.057817
2.0    0.001368
Name: proportion, dtype: float64

Kitchen AbvGr
1    0.954188
2    0.044103
0    0.001026
3    0.000684
Name: proportion, dtype: float64

3Ssn Porch
0      0.987350
168    0.001026
153    0.001026
144    0.000684
216    0.000684
180    0.000684
224    0.000342
255    0.00034

In [17]:
# look for 'unbalanced' numerics, greater than 80%, since these are 0 or not 0, swith to bool
for col in df.select_dtypes(include=np.number).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.80:
    print(df[col].value_counts(normalize=True))
    print()
    if len(df[col].unique()) > 2:
      df[col] = np.where(df[col] > 0, 1, 0)
    df[col] = df[col].astype('boolean')

BsmtFin SF 2
0.0       0.879959
294.0     0.001710
180.0     0.001710
162.0     0.001026
182.0     0.001026
            ...   
1085.0    0.000342
682.0     0.000342
441.0     0.000342
163.0     0.000342
1120.0    0.000342
Name: proportion, Length: 274, dtype: float64

Enclosed Porch
0      0.843077
112    0.007521
96     0.004444
144    0.003761
192    0.003419
         ...   
272    0.000342
429    0.000342
67     0.000342
132    0.000342
23     0.000342
Name: proportion, Length: 183, dtype: float64



In [18]:
# look for 'unbalanced' numerics, greater than 59%
for col in df.select_dtypes(include=np.number).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.59:
    print(df[col].value_counts(normalize=True))
    print()

# I'll leave Half Bath in, just because I'd personally want it in the model.

# Mas Vnr Area (Masonry Veneer area) is 60% zero, dropping this one
# Masonry veneer is usually associated with 'wealthy' looking houses
# It would be worth holding onto this for non-basic models.
df = safe_drop(df, ['Mas Vnr Area'])

Mas Vnr Area
0.0      0.601999
120.0    0.005169
176.0    0.004480
200.0    0.004480
216.0    0.004135
           ...   
293.0    0.000345
653.0    0.000345
630.0    0.000345
382.0    0.000345
443.0    0.000345
Name: proportion, Length: 442, dtype: float64

Half Bath
0    0.630085
1    0.361368
2    0.008547
Name: proportion, dtype: float64



In [19]:
# recheck missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2925 entries, 0 to 2929
Data columns (total 37 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   MS SubClass            2925 non-null   int64   
 1   Lot Frontage           2435 non-null   float64 
 2   Lot Area               2925 non-null   int64   
 3   Overall Qual           2925 non-null   int64   
 4   Overall Cond           2925 non-null   int64   
 5   Year Built             2925 non-null   int64   
 6   Year Remod/Add         2925 non-null   int64   
 7   Foundation             2925 non-null   category
 8   BsmtFin SF 1           2924 non-null   float64 
 9   BsmtFin SF 2           2925 non-null   boolean 
 10  Bsmt Unf SF            2924 non-null   float64 
 11  Total Bsmt SF          2924 non-null   float64 
 12  1st Flr SF             2925 non-null   int64   
 13  2nd Flr SF             2925 non-null   int64   
 14  Gr Liv Area            2925 non-null   int64 

In [20]:
# median replacement for all remaining numeric types
num_cols_with_na = df.select_dtypes(include=np.number).columns[df.select_dtypes(include=np.number).isnull().any()].tolist()
for col in num_cols_with_na:
    df[col] = df[col].fillna(df[col].median())

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2925 entries, 0 to 2929
Data columns (total 37 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   MS SubClass            2925 non-null   int64   
 1   Lot Frontage           2925 non-null   float64 
 2   Lot Area               2925 non-null   int64   
 3   Overall Qual           2925 non-null   int64   
 4   Overall Cond           2925 non-null   int64   
 5   Year Built             2925 non-null   int64   
 6   Year Remod/Add         2925 non-null   int64   
 7   Foundation             2925 non-null   category
 8   BsmtFin SF 1           2925 non-null   float64 
 9   BsmtFin SF 2           2925 non-null   boolean 
 10  Bsmt Unf SF            2925 non-null   float64 
 11  Total Bsmt SF          2925 non-null   float64 
 12  1st Flr SF             2925 non-null   int64   
 13  2nd Flr SF             2925 non-null   int64   
 14  Gr Liv Area            2925 non-null   int64 

## Conclusion
We made (and documented) a bunch of choices in how we cleaned and simplified our data. We've made these choices knowing that we intend to use multiple linear regression techniques. If we planned on using other techniques (e.g., tree-based regression that we will cover soon) and were interested only in prediction, we'd leave in more and clean a bit less.

### Where This Work Goes Next
All of the cleaning steps we walked through here have been packaged into a reusable module called `ames_cleaning.py`. In Part 2, we will load that module and call `load_and_clean_ames()`, which applies every decision we just made, so we can focus on building and evaluating models without re-running 130 lines of cleaning code. The function exists to keep the modeling notebooks focused, but everything it does is exactly what we practiced here.